# Publication figures: group mixture linear

Run cells **top to bottom** once per checkpoint (section **1** defines `USER_HOME`, `PROJECT_DIR`, imports, and plotting — run it before any figure cell). All plot titles, axis labels, and legend entries are driven by **knob blocks** at the start of each figure cell (search for `# ---- knobs`).

**Paper order:** (1) environment → (2) config + MSE by position + baselines → (3) unit-vector probe → (4) MSE by position vs target component → (5) padding / context ablation + optional baselines → (6) attention maps → (7) head patch importance → (8) prediction lens (logit-style).

Extend later: add more trials, sweeps, or appendix figures in new sections; keep knobs at the top of each cell.

## 1. Environment

Same role as the **first setup cells** in `eval.ipynb`, kept short: **paths** (`PROJECT_DIR`, `SRC_DIR`), **conda / Python** sanity, **legacy protobuf** workaround and optional pin, optional **pip** fallback, **IPython** `%matplotlib inline` and `%autoreload`, quick **import smoke**, then publication matplotlib/seaborn defaults and **`SAVE_FIGURES`**.

**Paths:** `PROJECT_DIR` defaults to `~/icl-time-series` (override with env `ICL_TS_PROJECT`). **`SRC_DIR`** resolves in order: env `ICL_TS_SRC` if it contains `eval.py`, else `cwd`, else `PROJECT_DIR/src`, else `~/icl-time-series/src`. Checkpoint paths in section 2 still use `~/models/<task>/` unless you change that cell.

In [ ]:
# ---- knobs (section 1) ----
SAVE_FIGURES = False  # set True to write PNGs for every figure cell
FIGURE_DIR = "figures_pub"  # relative to cwd, or set an absolute path
FIGURE_DPI = 300

# eval.ipynb-style: default repo root ~/icl-time-series (override with env ICL_TS_PROJECT).
AUTO_PIP_INSTALL_MISSING = False  # last resort; prefer conda / environment.yml
AUTO_PIN_PROTOBUF_LEGACY = False  # True: best-effort pip protobuf<=3.20.3 (old wandb/protobuf stacks)
ENABLE_AUTORELOAD = True  # False for a frozen "Run All" publication pass
ENABLE_MATPLOTLIB_INLINE = True  # %matplotlib inline in notebooks

import os
import sys
import json
import copy
import subprocess

USER_HOME = os.path.expanduser("~")
PROJECT_DIR = os.environ.get("ICL_TS_PROJECT", "").strip() or os.path.join(USER_HOME, "icl-time-series")
ICL_TS_SRC = os.environ.get("ICL_TS_SRC", "").strip()

# Harmless on modern stacks; matches eval.ipynb workaround for older protobuf / wandb.
os.environ.setdefault("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION", "python")

if AUTO_PIN_PROTOBUF_LEGACY:
    try:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "protobuf<=3.20.3", "--quiet", "--user"],
            check=False,
            timeout=180,
        )
    except Exception as _e:
        print("AUTO_PIN_PROTOBUF_LEGACY: skipped ({})".format(_e))


def _resolve_src_dir():
    if ICL_TS_SRC and os.path.isfile(os.path.join(ICL_TS_SRC, "eval.py")):
        return os.path.abspath(ICL_TS_SRC)
    here = os.path.abspath(os.getcwd())
    for p in (here, os.path.join(PROJECT_DIR, "src"), os.path.join(USER_HOME, "icl-time-series", "src")):
        if os.path.isfile(os.path.join(p, "eval.py")):
            return p
    return here


SRC_DIR = _resolve_src_dir()
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

_conda = os.environ.get("CONDA_PREFIX")
print("SRC_DIR:", SRC_DIR)
print("PROJECT_DIR:", PROJECT_DIR)
if _conda:
    print("conda:", _conda)
else:
    print("conda: (none) — activate your env if imports fail (see environment.yml).")
print("python:", sys.executable)

_ip = None
try:
    from IPython import get_ipython

    _ip = get_ipython()
except Exception:
    pass
if _ip is not None:
    if ENABLE_MATPLOTLIB_INLINE:
        _ip.run_line_magic("matplotlib", "inline")
    if ENABLE_AUTORELOAD:
        _ip.run_line_magic("load_ext", "autoreload")
        _ip.run_line_magic("autoreload", "2")
    _ipy_parts = []
    if ENABLE_MATPLOTLIB_INLINE:
        _ipy_parts.append("matplotlib inline")
    if ENABLE_AUTORELOAD:
        _ipy_parts.append("autoreload 2")
    if _ipy_parts:
        print("IPython:", ", ".join(_ipy_parts))
elif ENABLE_AUTORELOAD or ENABLE_MATPLOTLIB_INLINE:
    print("IPython not available; skipped inline/autoreload.")

if AUTO_PIP_INSTALL_MISSING:
    _specs = [
        ("numpy", "numpy"),
        ("torch", "torch"),
        ("pandas", "pandas"),
        ("matplotlib", "matplotlib"),
        ("seaborn", "seaborn"),
        ("tqdm", "tqdm"),
        ("pyyaml", "yaml"),
        ("munch", "munch"),
    ]
    for pip_name, imp in _specs:
        try:
            __import__(imp)
        except ImportError:
            print("pip install:", pip_name)
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", pip_name, "--quiet"],
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL,
            )

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import pandas as pd
from tqdm.auto import tqdm

from eval import get_model_from_run
from samplers import get_data_sampler
from tasks import get_task_sampler
from oracle_utils import compute_all_group_mixture_baselines_mse_by_position

try:
    import eval as _eval_mod  # noqa: F401
    print("OK: eval from", os.path.dirname(_eval_mod.__file__))
except ImportError as e:
    raise ImportError(
        "Could not import project modules. cd to repo src/, or set ICL_TS_SRC to that path, "
        "or activate your conda env (see environment.yml)."
    ) from e

# Compact import check (eval.ipynb printed each module; we only warn on failure).
_missing_core = []
for _mod in ("yaml", "munch"):
    try:
        __import__(_mod)
    except ImportError:
        _missing_core.append(_mod)
if _missing_core:
    print("WARNING: missing modules:", _missing_core, "(use conda / environment.yml or AUTO_PIP_INSTALL_MISSING)")

sns.set_theme(context="paper", style="whitegrid", font_scale=1.15)
mpl.rcParams.update({
    "figure.figsize": (5.5, 3.5),
    "figure.dpi": 120,
    "savefig.dpi": FIGURE_DPI,
    "savefig.bbox": "tight",
    "axes.labelsize": 11,
    "axes.titlesize": 12,
    "legend.fontsize": 9,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "lines.linewidth": 1.8,
    "lines.markersize": 4,
})

os.makedirs(FIGURE_DIR, exist_ok=True)
_fig_counter = {"n": 0}


def save_pub_figure(fig, stem: str):
    """Save if SAVE_FIGURES; stem should be filesystem-safe."""
    if not SAVE_FIGURES:
        return
    _fig_counter["n"] += 1
    safe = stem.replace(" ", "_").replace("/", "-")
    path = os.path.join(FIGURE_DIR, f"{_fig_counter['n']:02d}_{safe}.png")
    fig.savefig(path)
    print(f"Saved: {path}")


def bootstrap_mean_ci_clip(curves, n_boot=400, alpha=0.05, clip_low=0.0):
    """
    curves: (n_trials, n_positions) -- e.g. MSE curves per trial.
    Returns mean, lower, upper with lower clipped at clip_low (MSE cannot be negative).
    """
    curves = np.asarray(curves, dtype=float)
    if curves.ndim == 1:
        curves = curves.reshape(1, -1)
    n_t, n_p = curves.shape
    mean = np.nanmean(curves, axis=0)
    if n_t < 3:
        return mean, mean, mean
    rng = np.random.default_rng(0)
    boots = np.empty((n_boot, n_p), dtype=float)
    for b in range(n_boot):
        idx = rng.integers(0, n_t, size=n_t)
        boots[b] = np.nanmean(curves[idx], axis=0)
    lo = np.quantile(boots, alpha / 2, axis=0)
    hi = np.quantile(boots, 1 - alpha / 2, axis=0)
    lo = np.maximum(lo, clip_low)
    hi = np.maximum(hi, clip_low)
    return mean, lo, hi


def task_kwargs_from_conf(conf):
    tk = getattr(conf.training, "task_kwargs", None)
    if tk is None:
        return {}
    return dict(tk) if isinstance(tk, dict) else dict(tk.__dict__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(
    "device:", device,
    "| SAVE_FIGURES:", SAVE_FIGURES,
    "| figures:", os.path.abspath(FIGURE_DIR),
)


## 2. Configuration + MSE by position (transformer vs baselines)

Load **`config.yaml`** from the run directory (same as training). **K, C, T** come from `task_kwargs` and the sampler — no hard-coded task geometry.

Edit **`RUN_ID`** and optional **`CHECKPOINT_STEP`** (`-1` = `state.pt`). Baseline **display names** and **which curves to plot** are lists/dicts in the knob block.

In [ ]:
# ---- knobs (section 2: load + main MSE-by-position figure) ----
# USER_HOME is set in section 1 (same as eval.ipynb: ~/…)
RUN_TASK_SUBDIR = "group_mixture_linear"  # folder under ~/models/
RUN_ID = "REPLACE_WITH_RUN_UUID"
CHECKPOINT_STEP = -1  # -1: state.pt; else state_STEP.pt / model_STEP.pt

N_TRIALS_MSE = 30
BATCH_SIZE_EVAL = None  # None -> use training batch_size from config
BOOTSTRAP_CI = True
BOOTSTRAP_ALPHA = 0.05

# Which oracle/baseline keys to draw (order = legend order). Edit freely.
BASELINE_KEYS_ORDER = [
    "ground_truth",
    "true_w_unknown_assignment_bayesian",
    "bayesian_mixture",
    "pure_ls_target",
    "hybrid_bayesian_ls",
]
# Optional prettier labels (key -> string). Leave empty to use defaults below.
BASELINE_LABEL_OVERRIDES = {
    # "bayesian_mixture": "Your label here",
}
_BASELINE_DEFAULT_LABELS = {
    "ground_truth": "Oracle (known w, known assignment)",
    "true_w_unknown_assignment_bayesian": "Bayes (known w, unknown assignment)",
    "bayesian_mixture": "Mixture Bayes (LS context + Bayes target)",
    "pure_ls_target": "LS target only",
    "hybrid_bayesian_ls": "Hybrid Bayes–LS",
}

FIG_TITLE_MSE = "MSE by position"
FIG_XLABEL_MSE = "Sequence index"
FIG_YLABEL_MSE = "Mean squared error"
TRANSFORMER_LEGEND = "Transformer"

run_dir = os.path.join(USER_HOME, "models", RUN_TASK_SUBDIR)
run_path = os.path.join(run_dir, RUN_ID)
if "REPLACE" in RUN_ID:
    raise ValueError("Set RUN_ID to your trained run UUID folder name.")

model, conf = get_model_from_run(run_path, step=CHECKPOINT_STEP)
model.eval().to(device)

task_kwargs = task_kwargs_from_conf(conf)
n_dims = int(conf.model.n_dims)
bs = int(BATCH_SIZE_EVAL or conf.training.batch_size)

# Main eval: match training layout (predict_target_only from config unless overridden here)
OVERRIDE_PREDICT_TARGET_ONLY = None  # e.g. True; None = use config
if OVERRIDE_PREDICT_TARGET_ONLY is not None:
    task_kwargs = dict(task_kwargs)
    task_kwargs["predict_target_only"] = OVERRIDE_PREDICT_TARGET_ONLY

data_sampler = get_data_sampler(conf.training.data, n_dims=n_dims, **task_kwargs)
task_sampler = get_task_sampler(
    conf.training.task,
    n_dims,
    bs,
    num_tasks=getattr(conf.training, "num_tasks", None),
    **task_kwargs,
)
seq_struct = data_sampler.get_sequence_structure()
predict_inds = seq_struct["predict_inds"]
n_points = seq_struct["total_length"]
K = int(data_sampler.n_components)
C = int(data_sampler.contexts_per_component)
T_tgt = int(data_sampler.target_cluster_context_points)
context_length = K * C

metric = None
trial_curves = []
baseline_accum = {k: None for k in BASELINE_KEYS_ORDER}

for trial in tqdm(range(N_TRIALS_MSE), desc="MSE trials"):
    seed0 = 50_000 + trial * max(bs, 1)
    xs = data_sampler.sample_xs(
        n_points=n_points,
        b_size=bs,
        seeds=list(range(seed0, seed0 + bs)),
    )
    task = task_sampler(
        components=data_sampler.current_components,
        component_assignments=data_sampler.component_assignments,
    )
    ys = task.evaluate(xs)
    if metric is None:
        metric = task.get_metric()

    with torch.no_grad():
        if len(predict_inds) == ys.shape[1]:
            pred = model(xs.to(device), ys.to(device))
        else:
            pred = model(
                xs.to(device),
                ys.to(device),
                inds=predict_inds,
                sequence_structure=seq_struct,
            )

    if len(predict_inds) == ys.shape[1]:
        err = metric(pred, ys.to(pred.device))
    else:
        err = metric(pred, ys.to(pred.device)[:, predict_inds])
    err = err.detach().float().cpu().numpy()
    v = np.full((n_points,), np.nan, dtype=float)
    if err.ndim == 0:
        val = float(err)
        if len(predict_inds) == n_points:
            v[:] = val
        else:
            for idx in predict_inds:
                v[idx] = val
    elif err.ndim == 1:
        sub = np.atleast_1d(err).astype(float).ravel()
        for j in range(min(len(sub), len(predict_inds))):
            v[predict_inds[j]] = float(sub[j])
    else:
        sub = err.mean(axis=0).astype(float).ravel()
        for j in range(min(len(sub), len(predict_inds))):
            v[predict_inds[j]] = float(sub[j])
    trial_curves.append(v)

    scale_v = float(getattr(task, "scale", getattr(data_sampler, "scale", 1.0)))
    noise_v = float(getattr(task, "noise_std", getattr(data_sampler, "noise_std", 0.0)))
    mb = compute_all_group_mixture_baselines_mse_by_position(
        xs.cpu(), ys.cpu(),
        data_sampler.current_components,
        data_sampler.component_assignments,
        K, C, T_tgt, scale_v,
        target_noise_std=noise_v,
    )
    for k in BASELINE_KEYS_ORDER:
        if k not in mb:
            continue
        a = mb[k].astype(float)
        baseline_accum[k] = a if baseline_accum[k] is None else baseline_accum[k] + a

trial_curves = np.stack(trial_curves, axis=0)
mean_mse, lo_mse, hi_mse = bootstrap_mean_ci_clip(trial_curves, alpha=BOOTSTRAP_ALPHA)
if not BOOTSTRAP_CI:
    lo_mse = hi_mse = mean_mse

x_pos = np.arange(n_points)
fig, ax = plt.subplots(1, 1, figsize=(6.5, 4.0))
_band_ok = np.isfinite(mean_mse) & np.isfinite(lo_mse) & np.isfinite(hi_mse)
if BOOTSTRAP_CI and _band_ok.any():
    ax.fill_between(
        x_pos, lo_mse, hi_mse, where=_band_ok,
        color="C0", alpha=0.2, label="Transformer CI",
    )
ax.plot(x_pos, mean_mse, color="C0", marker="o", label=TRANSFORMER_LEGEND)

for k in BASELINE_KEYS_ORDER:
    if baseline_accum[k] is None:
        continue
    curve = baseline_accum[k] / N_TRIALS_MSE
    lab = BASELINE_LABEL_OVERRIDES.get(k, _BASELINE_DEFAULT_LABELS.get(k, k))
    ax.plot(x_pos, curve, ls="--", label=lab)

if context_length < n_points:
    ax.axvline(context_length - 0.5, color="0.35", ls=":", lw=1.2, label="Context / target boundary")
ax.set_title(FIG_TITLE_MSE)
ax.set_xlabel(FIG_XLABEL_MSE)
ax.set_ylabel(FIG_YLABEL_MSE)
ax.legend(loc="best", frameon=True)
plt.tight_layout()
save_pub_figure(fig, "mse_by_position_baselines")
plt.show()

print("K, C, T_tgt, n_points:", K, C, T_tgt, n_points)

## 3. Unit-vector probe (target cluster; first point by default)

Mechanistic comparison at one target-cluster index. **Knobs** control offset, component, and plot labels.

In [ ]:
# ---- knobs (section 3) ----
PROBE_TARGET_COMPONENT = 0
PROBE_TARGET_CLUSTER_OFFSET = 0  # 0 = first point in target cluster
PROBE_FIXED_CLUSTER = None  # None -> [0,1,...,K-1]
PROBE_RANDOM_COUNT = 12
PROBE_SEED = 2027

FIG_TITLE_UNIT = "Unit-vector probe vs true weights"
FIG_TITLE_SCATTER = "Predictions vs counterfactual component outputs"

if conf.training.task != "group_mixture_linear":
    raise RuntimeError("This notebook is for group_mixture_linear.")

torch.manual_seed(PROBE_SEED)
np.random.seed(PROBE_SEED)

probe_kw = dict(task_kwargs)
probe_kw["predict_target_only"] = False
ds_p = get_data_sampler(conf.training.data, n_dims=n_dims, **probe_kw)
ts_p = get_task_sampler(conf.training.task, n_dims, 1, num_tasks=getattr(conf.training, "num_tasks", None), **probe_kw)
ss_p = ds_p.get_sequence_structure()
Ttot = ss_p["total_length"]
Kp = ds_p.n_components
Cp = ds_p.contexts_per_component
ctx0 = Kp * Cp
probe_idx = ctx0 + int(PROBE_TARGET_CLUSTER_OFFSET)
if probe_idx < ctx0 or probe_idx >= Ttot:
    raise ValueError("PROBE_TARGET_CLUSTER_OFFSET out of range")

fixed_ctx = list(range(Kp)) if PROBE_FIXED_CLUSTER is None else PROBE_FIXED_CLUSTER
xs_base = ds_p.sample_xs(
    n_points=Ttot, b_size=1,
    fixed_cluster_assignments=fixed_ctx,
    fixed_target_component=PROBE_TARGET_COMPONENT,
    seeds=[12345],
)
task_p = ts_p(components=ds_p.current_components, component_assignments=ds_p.component_assignments)
scale_p = float(getattr(task_p, "scale", 1.0))
w_all = (ds_p.current_components[0, :, :, 0].cpu() * scale_p).numpy()  # (K, d)
true_w = w_all[PROBE_TARGET_COMPONENT]

md = next(model.parameters()).device
inds_one = [probe_idx]

def fwd_one(xs_t, ys_t):
    with torch.no_grad():
        return model(xs_t.to(md), ys_t.to(md), inds=inds_one, sequence_structure=ss_p)

def eval_x(xv):
    xs_q = xs_base.clone()
    xs_q[0, probe_idx, :] = xv
    ys_q = task_p.evaluate(xs_q)
    pred = fwd_one(xs_q, ys_q)
    return float(pred[0, 0].cpu()), float(ys_q[0, probe_idx].cpu())

rows = []
for i in range(PROBE_RANDOM_COUNT):
    xq = torch.randn(n_dims)
    pr, tr = eval_x(xq)
    y_if = {f"y_if_{k}": float((xq.numpy() * w_all[k]).sum()) for k in range(Kp)}
    row = {"i": i, "pred": pr, "true": tr, "|err|": abs(pr - tr)}
    row.update(y_if)
    rows.append(row)

df_sc = pd.DataFrame(rows)

unit_pred = []
for j in range(n_dims):
    e = torch.zeros(n_dims)
    e[j] = 1.0
    p, t = eval_x(e)
    unit_pred.append(p)
unit_pred = np.array(unit_pred)

dims = np.arange(n_dims)
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8))
axes[0].scatter(df_sc["true"], df_sc["pred"], label="target y", zorder=3)
for k in range(Kp):
    if k == PROBE_TARGET_COMPONENT:
        continue
    c = f"y_if_{k}"
    if c in df_sc:
        axes[0].scatter(df_sc[c], df_sc["pred"], marker="x", alpha=0.85, label=f"counterfactual comp {k}")
mn = min(df_sc["pred"].min(), df_sc["true"].min(), *[df_sc[f"y_if_{k}"].min() for k in range(Kp) if f"y_if_{k}" in df_sc])
mx = max(df_sc["pred"].max(), df_sc["true"].max(), *[df_sc[f"y_if_{k}"].max() for k in range(Kp) if f"y_if_{k}" in df_sc])
axes[0].plot([mn, mx], [mn, mx], "k--", lw=1, alpha=0.55)
axes[0].set_title(FIG_TITLE_SCATTER)
axes[0].set_xlabel("Reference y")
axes[0].set_ylabel("Predicted y")
axes[0].legend(fontsize=7, loc="best")

axes[1].plot(dims, unit_pred, "o-", label="pred @ e_j")
axes[1].plot(dims, true_w, "s--", label=f"true w comp {PROBE_TARGET_COMPONENT}")
for k in range(Kp):
    if k == PROBE_TARGET_COMPONENT:
        continue
    axes[1].plot(dims, w_all[k], "--", marker="x", lw=1.2, label=f"true w comp {k}")
axes[1].set_xticks(dims)
axes[1].set_xticklabels([str(int(j)) for j in dims])
axes[1].set_xlabel("Dimension")
axes[1].set_ylabel("Coefficient")
axes[1].set_title(FIG_TITLE_UNIT)
axes[1].legend(fontsize=7, loc="best")
plt.tight_layout()
save_pub_figure(fig, "unit_vector_probe")
plt.show()

## 4. MSE by position vs target component (symmetry / appendix)

Same context cluster assignment **0..K-1**, vary **fixed target component**. Uses **predict-all** structure from sampler; per-position MSE averaged over trials. **Knobs** rename figure and trial count.

In [ ]:
# ---- knobs (section 4) ----
N_TRIALS_TARGET_SWEEP = 8
TARGET_SWEEP_BS = min(8, bs)
FIG_TITLE_TARGET_SWEEP = "MSE by position for each fixed target component"
FIXED_CTX_FOR_SWEEP = None  # None -> [0,...,K-1]

sweep_kw = dict(task_kwargs)
sweep_kw["predict_target_only"] = False
ds_s = get_data_sampler(conf.training.data, n_dims=n_dims, **sweep_kw)
ts_s = get_task_sampler(conf.training.task, n_dims, TARGET_SWEEP_BS, num_tasks=getattr(conf.training, "num_tasks", None), **sweep_kw)
ss_s = ds_s.get_sequence_structure()
T_s = ss_s["total_length"]
K_s = ds_s.n_components
met_ts = None

fig, ax = plt.subplots(1, 1, figsize=(7.0, 4.2))
markers = "os^DvP"
ctx_assign = list(range(K_s)) if FIXED_CTX_FOR_SWEEP is None else FIXED_CTX_FOR_SWEEP

for tau in range(K_s):
    curves_tau = []
    for trial in range(N_TRIALS_TARGET_SWEEP):
        s0 = 70_000 + trial * 1000 + tau * 17
        xs = ds_s.sample_xs(T_s, TARGET_SWEEP_BS, fixed_cluster_assignments=ctx_assign, fixed_target_component=tau, seeds=list(range(s0, s0 + TARGET_SWEEP_BS)))
        task = ts_s(components=ds_s.current_components, component_assignments=ds_s.component_assignments)
        ys = task.evaluate(xs)
        if met_ts is None:
            met_ts = task.get_metric()
        mse_pos = np.zeros(T_s, dtype=float)
        for pos in range(T_s):
            with torch.no_grad():
                pr = model(xs.to(device), ys.to(device), inds=[pos], sequence_structure=ss_s)
            er = met_ts(pr, ys.to(pr.device)[:, pos : pos + 1]).detach().float().cpu().numpy()
            mse_pos[pos] = float(np.mean(er))
        curves_tau.append(mse_pos)
    curves_tau = np.stack(curves_tau, axis=0)
    mean_c, lo_c, hi_c = bootstrap_mean_ci_clip(curves_tau, alpha=BOOTSTRAP_ALPHA)
    if not BOOTSTRAP_CI:
        lo_c = hi_c = mean_c
    x_ = np.arange(T_s)
    ax.fill_between(x_, lo_c, hi_c, alpha=0.15)
    ax.plot(x_, mean_c, marker=markers[tau % len(markers)], label=f"Target component {tau}")

ax.axvline(K_s * ds_s.contexts_per_component - 0.5, color="0.35", ls=":", lw=1.2, label="Context end")
ax.set_title(FIG_TITLE_TARGET_SWEEP)
ax.set_xlabel(FIG_XLABEL_MSE)
ax.set_ylabel(FIG_YLABEL_MSE)
ax.legend(loc="best", fontsize=8, ncol=2)
plt.tight_layout()
save_pub_figure(fig, "mse_by_position_target_sweep")
plt.show()

## 5. Context padding: MSE by position + optional baselines

For each **i** contexts per cluster (rest zero-padded), full sequence length unchanged; **target cluster unchanged**. Per-position MSE of **(pred - ys_pad)** with one forward per position index (memory-safe).

Set **`INCLUDE_BASELINES_PADDING`** to a subset of baseline keys or `[]` to skip (baselines are expensive per i).

In [ ]:
# ---- knobs (section 5) ----
PADDING_BS = min(8, bs)
INCLUDE_BASELINES_PADDING = []  # e.g. ["ground_truth", "bayesian_mixture"] or [] to skip
FIG_TITLE_PAD = "MSE by position under context padding"
LINE_LABEL_I = "contexts per cluster = {i}"  # formatted with .format(i=i)

pad_kw = dict(task_kwargs)
pad_kw["predict_target_only"] = False
ds_f = get_data_sampler(conf.training.data, n_dims=n_dims, **pad_kw)
ts_f = get_task_sampler(conf.training.task, n_dims, PADDING_BS, num_tasks=getattr(conf.training, "num_tasks", None), **pad_kw)
ss_f = ds_f.get_sequence_structure()
T_f = ss_f["total_length"]
K_f = ds_f.n_components
C_f = ds_f.contexts_per_component
ctx_len_f = K_f * C_f

xs_full = ds_f.sample_xs(T_f, PADDING_BS, seeds=list(range(3000, 3000 + PADDING_BS)))
task0 = ts_f(components=ds_f.current_components, component_assignments=ds_f.component_assignments)
ys_full = task0.evaluate(xs_full)

def mse_curve_padded(i_ctx):
    xs_p = xs_full.clone()
    ys_p = ys_full.clone()
    for k in range(K_f):
        st = k * C_f
        xs_p[:, st + i_ctx : st + C_f, :] = 0.0
        ys_p[:, st + i_ctx : st + C_f] = 0.0
    mse_pos = np.zeros(T_f, dtype=float)
    for pos in range(T_f):
        with torch.no_grad():
            pr = model(xs_p.to(device), ys_p.to(device), inds=[pos], sequence_structure=ss_f)
        tgt = ys_p[:, pos : pos + 1].to(pr.device)
        mse_pos[pos] = ((pr - tgt) ** 2).mean().item()
    return mse_pos, xs_p, ys_p

fig, ax = plt.subplots(1, 1, figsize=(7.2, 4.2))
positions = np.arange(T_f)
baseline_pad_curves = {bk: [] for bk in INCLUDE_BASELINES_PADDING}

for i_ctx in tqdm(range(1, C_f + 1), desc="padding i"):
    mse_pos, xs_p, ys_p = mse_curve_padded(i_ctx)
    ax.plot(positions, mse_pos, label=LINE_LABEL_I.format(i=i_ctx))
    scale_v = float(getattr(task0, "scale", getattr(ds_f, "scale", 1.0)))
    noise_v = float(getattr(task0, "noise_std", getattr(ds_f, "noise_std", 0.0)))
    for bk in INCLUDE_BASELINES_PADDING:
        mb = compute_all_group_mixture_baselines_mse_by_position(
            xs_p.cpu(), ys_p.cpu(), ds_f.current_components, ds_f.component_assignments,
            K_f, C_f, int(ds_f.target_cluster_context_points), scale_v, target_noise_std=noise_v,
        )
        baseline_pad_curves[bk].append(mb[bk].astype(float))

if ctx_len_f < T_f:
    ax.axvline(ctx_len_f - 0.5, color="0.35", ls=":", lw=1.2, label="Context end")

for bk in INCLUDE_BASELINES_PADDING:
    curves_b = np.stack(baseline_pad_curves[bk], axis=0)  # (n_i, T_f)
    mean_b = curves_b.mean(axis=0)
    lab = BASELINE_LABEL_OVERRIDES.get(bk, _BASELINE_DEFAULT_LABELS.get(bk, bk))
    ax.plot(positions, mean_b, ls="--", lw=1.5, label=f"{lab} (mean over i)")

ax.set_title(FIG_TITLE_PAD)
ax.set_xlabel(FIG_XLABEL_MSE)
ax.set_ylabel(FIG_YLABEL_MSE)
ax.legend(loc="best", fontsize=7, ncol=2)
plt.tight_layout()
save_pub_figure(fig, "padding_mse_by_position")
plt.show()

## 6. Attention maps by layer

**Knobs:** which layer(s), batch index, head average or single head. Extend later (e.g. facet by target component).

In [ ]:
# ---- knobs (section 6) ----
ATTN_LAYERS_TO_PLOT = [0, 2, 5, 8, 11]  # edit; must be < n_layer
ATTN_BATCH_IDX = 0
ATTN_AVERAGE_HEADS = True
ATTN_HEAD_IDX = 0
FIG_TITLE_ATTN = "Attention weights (query vs key)"

ds_a = get_data_sampler(conf.training.data, n_dims=n_dims, **task_kwargs)
ss_a = ds_a.get_sequence_structure()
T_a = ss_a["total_length"]
xs_a = ds_a.sample_xs(T_a, min(bs, 4), seeds=list(range(6000, 6000 + min(bs, 4))))
task_a = task_sampler(components=ds_a.current_components, component_assignments=ds_a.component_assignments)
ys_a = task_a.evaluate(xs_a)

with torch.no_grad():
    attns = model.get_attention(xs_a.to(device), ys_a.to(device), ss_a)
n_layer = len(attns)
layers = [ell for ell in ATTN_LAYERS_TO_PLOT if 0 <= ell < n_layer]
if not layers:
    layers = [min(5, n_layer - 1)]

n_pl = len(layers)
fig, axes = plt.subplots(1, n_pl, figsize=(3.2 * n_pl + 1, 3.8), constrained_layout=True)
if n_pl == 1:
    axes = [axes]
vmax = 0.0
mats = []
for ax, ell in zip(axes, layers):
    a = attns[ell][ATTN_BATCH_IDX]
    if ATTN_AVERAGE_HEADS:
        a = a.mean(dim=0).cpu().numpy()
    else:
        a = a[ATTN_HEAD_IDX].cpu().numpy()
    mats.append(a)
    vmax = max(vmax, float(a.max()))
for ax, a, ell in zip(axes, mats, layers):
    im = ax.imshow(a, cmap="Blues", aspect="auto", vmin=0.0, vmax=max(vmax, 1e-8))
    ax.set_title(f"L{ell}")
    ax.set_xlabel("key")
axes[0].set_ylabel("query")
fig.suptitle(FIG_TITLE_ATTN)
fig.colorbar(im, ax=axes, shrink=0.75, label="weight")
save_pub_figure(fig, "attention_layers")
plt.show()

## 7. Head patch importance (c_proj column ablation)

Same mechanism as exploratory `eval.ipynb`: temporarily zero **c_proj** columns for one head, measure MSE change at query. **Knobs** control layers scanned and batch size.

In [ ]:
# ---- knobs (section 7) ----
PATCH_BS = min(8, bs)
PATCH_LAYERS = None  # None -> all layers
FIG_TITLE_PATCH = "Head importance (Δ MSE under c_proj patch)"

cfg = model._backbone.config
nL, nH, nE = cfg.n_layer, cfg.n_head, cfg.n_embd
assert nE % nH == 0
hd = nE // nH
layers_use = list(range(nL)) if not PATCH_LAYERS else [j for j in PATCH_LAYERS if 0 <= j < nL]

ds_h = get_data_sampler(conf.training.data, n_dims=n_dims, **task_kwargs)
ss_h = ds_h.get_sequence_structure()
Th = ss_h["total_length"]
qh = 2 * (Th - 1)
xs_h = ds_h.sample_xs(Th, PATCH_BS, seeds=list(range(8000, 8000 + PATCH_BS)))
task_h = task_sampler(components=ds_h.current_components, component_assignments=ds_h.component_assignments)
ys_h = task_h.evaluate(xs_h)
yq = ys_h[:, -1].to(device)

def mse_query():
    with torch.no_grad():
        p = model(xs_h.to(device), ys_h.to(device), inds=[Th - 1], sequence_structure=ss_h).squeeze(-1)
    return float(((p - yq) ** 2).mean().item())

imp = np.zeros((len(layers_use), nH), dtype=float)
base_mse = mse_query()
blocks = model._backbone.h
for li, layer_idx in enumerate(tqdm(layers_use, desc="layers")):
    blk = blocks[layer_idx]
    W = blk.attn.c_proj.weight
    W0 = W.data.clone()
    for h in range(nH):
        c0, c1 = h * hd, (h + 1) * hd
        W.data[:, c0:c1] = 0.0
        imp[li, h] = mse_query() - base_mse
        W.data.copy_(W0)

fig, ax = plt.subplots(1, 1, figsize=(6.5 + 0.35 * nH, 3.8))
im = ax.imshow(imp, aspect="auto", cmap="magma")
ax.set_yticks(range(len(layers_use)))
ax.set_yticklabels([str(layers_use[i]) for i in range(len(layers_use))])
ax.set_xticks(range(nH))
ax.set_xlabel("head")
ax.set_ylabel("layer")
ax.set_title(FIG_TITLE_PATCH + f" (baseline MSE={base_mse:.4f})")
fig.colorbar(im, ax=ax, shrink=0.65, label="Δ MSE")
plt.tight_layout()
save_pub_figure(fig, "head_patch_importance")
plt.show()

## 8. Prediction lens (logit-style readout per layer)

Project residual stream at the **query x-token** through **`_read_out`** (and `ln_f` if present), per layer. **Knobs** control example count and title.

In [ ]:
# ---- knobs (section 8) ----
LENS_N_EX = 8
FIG_TITLE_LENS = "Prediction lens: readout vs layer"

ds_l = get_data_sampler(conf.training.data, n_dims=n_dims, **task_kwargs)
ss_l = ds_l.get_sequence_structure()
Tl = ss_l["total_length"]
qidx = 2 * (Tl - 1)
xs_l = ds_l.sample_xs(Tl, LENS_N_EX, seeds=list(range(9000, 9000 + LENS_N_EX)))
task_l = task_sampler(components=ds_l.current_components, component_assignments=ds_l.component_assignments)
ys_l = task_l.evaluate(xs_l)
true_y = ys_l[:, -1].cpu().numpy()

with torch.no_grad():
    hstates = model.get_residual_stream_by_layer(xs_l.to(device), ys_l.to(device), ss_l)
ln_f = getattr(model._backbone, "ln_f", None)
preds_by_layer = []
for h in hstates:
    hq = h[:, qidx, :]
    if ln_f is not None:
        hq = ln_f(hq)
    out = model._read_out(hq.unsqueeze(1)).squeeze(-1)
    preds_by_layer.append(out.detach().cpu().numpy())
P = np.stack(preds_by_layer, axis=0)  # (L+1, B)
layers_axis = np.arange(P.shape[0])

fig, ax = plt.subplots(1, 1, figsize=(6.8, 4.0))
for b in range(min(6, LENS_N_EX)):
    ax.plot(layers_axis, P[:, b], "o-", markersize=3, label=f"ex {b} (y*={true_y[b]:.2f})")
ax.axhline(0.0, color="0.45", ls="--", lw=1)
ax.set_xlabel("layer index (0 = embed)")
ax.set_ylabel("readout y")
ax.set_title(FIG_TITLE_LENS)
ax.legend(fontsize=7, loc="best")
plt.tight_layout()
save_pub_figure(fig, "prediction_lens")
plt.show()